

## load required pacakages

In [1]:
import os,re, json, csv, joblib, time
import numpy as np
import pandas as pd
import sklearn
from joblib import Parallel, delayed
from pathlib import Path
from Bio.Seq import Seq
from Bio import SeqIO
from difflib import SequenceMatcher
from Bio.Data import CodonTable
from evcouplings.compare import DistanceMap
from scipy.optimize import check_grad
from scipy.sparse import issparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,precision_recall_curve, auc,roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, Lasso, RidgeCV,RidgeClassifierCV,Ridge,LogisticRegressionCV

In [2]:
from data_loading import *
from model_training import train_ridge_model
from data_preprocessing import *

In [3]:
## ------------------------
## GENE-LEVEL PIPELINE FOR TB RESISTANCE PREDICTION
## Author: Mahbuba Tasmin
## Description: Annotated end-to-end pipeline for extracting aligned sequences,
## translating to protein, handling frameshifts, and encoding features for phenotype modeling.
## ------------------------

# Step 1: Initialize gene(s) of interest

In [4]:
# Step 1: Initialize gene(s) of interest

# Define the candidate gene
# You can modify 'gene_name' to run the pipeline for a different gene
gene_name='gyrB'
genes_of_interest=gene_name.split(',')
print("Gene in consideration", gene_name)

Gene in consideration gyrB


In [5]:
# Set random seed for reproducibility
seed_everything(42)

# Step 2: Load phenotype and gene metadata

In [6]:
# Load phenotype data and other initializations
phenotype_paths = ["../data/catalog/master_table_resistance.csv","../data/catalog/cryptic_dataset.csv"]

In [7]:
gene_details = pd.read_csv("../data/catalog/all_drug_genes_locations.csv")
# Filter the DataFrame for the genes of interest
filtered_df = gene_details [gene_details ['gene_name'].str.contains('|'.join(genes_of_interest), case=False, na=False)]
drug_name = filtered_df['drug_full'].values[0].upper()

uniprot = filtered_df['Uniprot'].values[0]
entry = filtered_df['Entry'].values[0]
print(f"The drug associated with the gene {gene_name} is: {drug_name}")

The drug associated with the gene gyrB is: LEVOFLOXACIN


In [8]:
# Load data
phenotype_data = load_phenotype_data(phenotype_paths,drug_name).reset_index()
print(phenotype_data.head(2))
phenotype_data=phenotype_data.drop(['level_0'], axis=1)
phenotype_data["Isolate_mapped"] = [path.split("/")[-1].split(".vcf")[0].split(".cut")[0] for path in phenotype_data.path]
phenotype_mapping = dict(zip(phenotype_data['Isolate_mapped'], phenotype_data[drug_name.upper()]))

Number of records in phenotype data: 6581
   level_0  index  Isolate AMIKACIN AMOXICILLIN AMOXICILLIN_CLAVULANATE  \
0        0   10.0  00R1566      NaN         NaN                     NaN   
1        1   14.0  01R0239        R         NaN                     NaN   

  CAPREOMYCIN CIPROFLOXACIN CLARITHROMYCIN CLOFAZIMINE  ... biosample  \
0           R           NaN            NaN         NaN  ...       NaN   
1           R           NaN              R         NaN  ...       NaN   

  internal                                               path  \
0      NaN  /n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...   
1      NaN  /n/data1/hms/dbmi/farhat/rollingDB/genomic_dat...   

  Isolate_original    accessions Unnamed: 0 UNIQUEID   ID ROLLINGDB_ID  \
0          00R1566    ERS2401251        NaN      NaN  NaN          NaN   
1          01R0239  SAMN05916424        NaN      NaN  NaN          NaN   

  BIOSAMPLE_ACCESSION  
0                 NaN  
1                 NaN  

[2 rows x 45 columns

/work/pi_annagreen_umass_edu/mahbuba/Data-Curation-for-MTB/protein-tasks/protein_translation/data_loading.py:11: DtypeWarning: Columns (3,4,7,8,9,12,17,20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


# Step 3: Extract nucleotide sequences using reference alignment and gene coordinates

In [9]:
#define the fasta directory
fasta_dir = "../data/combined_fasta_files"

In [10]:
# ------------------- Utilities ----------------------

def get_fasta_and_alignment(row, fasta_dir):
    file_path = os.path.join(fasta_dir, row['filename'])
    alignment = load_alignment(file_path)
    return file_path, alignment

def get_h37rv_reference(alignment):
    return alignment.select(sequences=[alignment.id_to_index["MT_H37Rv"]])

def extract_sequence_from_alignment(h37rv_alignment):
    return ''.join(h37rv_alignment[0][h37rv_alignment[0] != "-"])

def extract_with_padding(h37rv_numbers, h37rv_ref, alignment, cds_start, cds_end, upstream_pad=100, downstream_pad=100):
    padded_start = cds_start - upstream_pad
    padded_end   = cds_end + downstream_pad

    subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, padded_start, padded_end, alignment)
    h37rv_alignment = select_subset_alignment(h37rv_ref, padded_start, padded_end, h37rv_numbers[:, column_index])
    
    ungapped_seq = extract_sequence_from_alignment(h37rv_alignment)

    rel_start = upstream_pad
    rel_end   = rel_start + (cds_end - cds_start)
    return ungapped_seq[rel_start:rel_end], subset_alignment, h37rv_alignment


def extract_from_operon(h37rv_sequence_str, row, operon_start, special_shift=0):
    cds_start = int(row['start_position_on_the_genomic_accession'])
    cds_end   = int(row['end_position_on_the_genomic_accession'])

    rel_start = cds_start - operon_start + special_shift
    rel_end   = cds_end - operon_start + special_shift + 1
    return h37rv_sequence_str[rel_start:rel_end]

# ------------------- Main Loop ----------------------

subset_alignment = ''
h37rv_nongap_indices = []

# Load H37Rv genomic map once
h37rv_numbers = np.load("X_matrix_H37RV_coords_regenerated.npy")

for index, row in filtered_df.iterrows():
    gene_name = row['gene_name']
    print(f"\nProcessing: {gene_name} ({row['filename']})")

    # Load alignment
    file_path, alignment = get_fasta_and_alignment(row, fasta_dir)
    print(f"Gene shape: {alignment.matrix.shape}")

    # Extract reference
    h37rv_ref = get_h37rv_reference(alignment)

    # --- Special handling ---
    if gene_name in ['rpoB']:
        cds_start = int(row['start_position_on_the_genomic_accession'])
        cds_end   = int(row['end_position_on_the_genomic_accession'])
        operon_start, operon_end = cds_start - 100, cds_end + 100
        h37rv_sequence_str,subset_alignment, h37rv_alignment = extract_with_padding(h37rv_numbers, h37rv_ref, alignment, cds_start, cds_end)

        
    elif gene_name in ['embA', 'embB', 'embC']:
        # First extract full operon once (4239763–4249810)
        operon_start, operon_end = 4239763, 4249810
        subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, operon_start, operon_end, alignment)
        h37rv_alignment = select_subset_alignment(h37rv_ref, operon_start, operon_end, h37rv_numbers[:, column_index])
        h37rv_sequence_str = extract_sequence_from_alignment(h37rv_alignment)

        # Then extract gene from full operon
        shift = 1 if gene_name == 'embB' else 0
        h37rv_sequence_str = extract_from_operon(h37rv_sequence_str, row, operon_start, special_shift=shift)

    elif gene_name in ['gyrA', 'gyrB']:
        operon_start, operon_end = 5140, 9918
        subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, operon_start, operon_end, alignment)
        h37rv_alignment = select_subset_alignment(h37rv_ref, operon_start, operon_end, h37rv_numbers[:, column_index])
        h37rv_sequence_str = extract_sequence_from_alignment(h37rv_alignment)
        h37rv_sequence_str = extract_from_operon(h37rv_sequence_str, row, operon_start)

    else:
        start_index = int(row['start_position_on_the_genomic_accession'])
        end_index   = int(row['end_position_on_the_genomic_accession'])
        subset_alignment, column_index, _, _ = sort_gene_indices(h37rv_numbers, start_index, end_index, alignment)
        h37rv_alignment = select_subset_alignment(h37rv_ref, start_index, end_index, h37rv_numbers[:, column_index])
        h37rv_sequence_str = extract_sequence_from_alignment(h37rv_alignment)
    h37rv_nongap_indices = np.where(h37rv_alignment[0] != "-")[0]
    # -- Final reporting --
    print(f"Final extracted length: {len(h37rv_sequence_str)}")



Processing: gyrB (gyrBA.fasta)
Gene shape: (23277, 4788)
Processed subset alignment for gene start 5140 and end 9918 in column 14
Final extracted length: 2028


In [11]:
# Compare with MycoBrowser CDS (optional)
ref_seqs=pd.read_csv('../data/catalog/protein_sequences.csv')
ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values[0]
ref_genome =  ref_seqs.loc[ref_seqs['gene'] == gene_name, 'genome_sequence'].values[0].upper()  
match = SequenceMatcher(None, ref_genome ,h37rv_sequence_str).find_longest_match(0, len(h37rv_sequence_str), 0, len(ref_genome))
print(f"Longest match at ref_genome[{match.b}:{match.b + match.size}]")
print(f"Match length: {match.size} / {len(ref_genome)} ({match.size / len(ref_genome) * 100:.2f}%)")
print(f"Reference Sequence Length: {len(ref_genome)}")
print(f"Extracted Sequence Length: {len(h37rv_sequence_str)}")


Longest match at ref_genome[0:2028]
Match length: 2028 / 2028 (100.00%)
Reference Sequence Length: 2028
Extracted Sequence Length: 2028


In [12]:
#check alignment matrix shape
print(alignment.matrix.shape)
filenames = [path.split("/")[-1].split(".cut")[0] for path in subset_alignment.ids]
print("number of files", len(filenames))

filename_to_sequence = {}
for i, filename in enumerate(filenames):
    if filename not in filename_to_sequence and i < len(subset_alignment):
        filename_to_sequence[filename] = ''.join(subset_alignment[i])
filenames = list(filename_to_sequence.keys()) 

(23277, 4788)
number of files 23277


# Step 4: Prepare sequence-phenotype CSV output

In [13]:
# Gene coordinate offsets relative to operon or padded extraction
operon_coords = {
    'gyrB':  (5240, 7267, 5140),
    'gyrA':  (7302, 9818, 5140),
    'embC': (4239863, 4243147, 4239763),
    'embA': (4243233, 4246517, 4239763),
    'embB': (4246513, 4249810, 4239763),
    'fabg1': (1673440, 1674183, 1673340),
    'rpoB': (759806, 763325, 759706)  # padded extraction for rpoB
}

output_file = f"../data/sequence_data_csv/{gene_name}_{drug_name}_combined_sequence_data.csv"
data_list = []

for filename in filenames:
    if filename in phenotype_mapping and filename in filename_to_sequence:
        phenotype = phenotype_mapping[filename]
        sequence = filename_to_sequence[filename]

        if gene_name in operon_coords:
            gene_start, gene_end, operon_start = operon_coords[gene_name]
            start_offset = gene_start - operon_start
            end_offset   = gene_end   - operon_start + 1
            if gene_name == 'embB':
                start_offset += 1
                adjusted_indices = h37rv_nongap_indices[start_offset:end_offset]
                sequence = "".join(np.array(list(sequence))[adjusted_indices])
            else:
                # Apply slice to ungapped indices from H37Rv
                adjusted_indices = h37rv_nongap_indices[start_offset:end_offset]
                sequence = "".join(np.array(list(sequence))[adjusted_indices])
        else:
            # For non-operonic or simple genes
            sequence = "".join(np.array(list(sequence))[h37rv_nongap_indices])

        data_list.append([filename, sequence, phenotype, len(sequence)])

# Save the CSV
with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Filename", "Sequence", "Phenotype", "seq_len"])
    writer.writerows(data_list)

print(f"CSV file '{output_file}' saved successfully!")


CSV file '../data/sequence_data_csv/gyrB_LEVOFLOXACIN_combined_sequence_data.csv' saved successfully!


# Step 5: Translate to protein and handle frame disruptions

In [14]:
file_path="../data/sequence_data_csv/"+gene_name+"_"+ drug_name+ "_combined_sequence_data.csv"
gene_sequence_data=pd.read_csv(file_path)

In [15]:
file_path

'../data/sequence_data_csv/gyrB_LEVOFLOXACIN_combined_sequence_data.csv'

In [16]:
seq1 = gene_sequence_data['Sequence'][0] 
seq2 = ref_genome


# Find longest matching subsequence (not just substring)
matcher = SequenceMatcher(None, seq2, seq1)
match = matcher.find_longest_match(0, len(seq2), 0, len(seq1))

print(f"Longest match at ref_genome[{match.a}:{match.a + match.size}]")
print(f"Match length: {match.size} / {len(seq1)} ({match.size / len(seq1):.2%})")
start_index = ref_genome.find(gene_sequence_data['Sequence'][0])
print("If the match is from the start, index will be zero:", start_index)

Longest match at ref_genome[0:2028]
Match length: 2028 / 2028 (100.00%)
If the match is from the start, index will be zero: 0


In [17]:
def hamming_distance(seq1, seq2):
    return sum(c1 != c2 for c1, c2 in zip(seq1, seq2))

gene_sequence_data['hamming_dist'] = gene_sequence_data['Sequence'].apply(lambda x: hamming_distance(x, ref_genome))


In [18]:
# Custom translation function to deal with gaps, ambiguous bases, or internal stops
def translate_sequence_with_gaps(dna_seq, table="Standard", to_stop=False, handle_stops='remove', ref_protein_length=None):
    codon_table = CodonTable.unambiguous_dna_by_name[table]
    standard_table = codon_table.forward_table
    stop_codons = codon_table.stop_codons

    dna_seq = dna_seq.upper()
    protein_seq = []
    frameshift_mutations = False

    seq_len = len(dna_seq)
    i = 0
    cumulative_gap_count = 0

    while i + 3 <= seq_len:
        codon = dna_seq[i:i+3]
        if '-' in codon:
            # Codon contains gap(s)
            cumulative_gap_count += codon.count('-')
            protein_seq.append('-')
        elif re.search(r'[^ATCG]', codon):
            # Codon contains ambiguous nucleotide(s)
            protein_seq.append('X')
        else:
            if codon in stop_codons:
                if to_stop:
                    break
                else:
                    if handle_stops == 'remove':
                        pass  # Ignore stop codon
                    elif handle_stops == 'replace':
                        protein_seq.append('X')  # Replace stop codon with X
                    else:
                        protein_seq.append('*')  # Keep stop codon
            else:
                amino_acid = standard_table.get(codon)
                protein_seq.append(amino_acid)
        i += 3

    # Handle remaining nucleotides at the end
    if i < seq_len:
        remaining = dna_seq[i:]
        if '-' in remaining or re.search(r'[^ATCG]', remaining):
            protein_seq.append('-')
        else:
            # Remaining nucleotides less than a codon length (ignore instead of flagging)
            pass

    protein_str = ''.join(protein_seq)

    # **Fix: Prevent unnecessary frameshift flagging**
    if ref_protein_length is not None:
        translated_length = len(protein_seq)

        # Allow small mismatches (e.g., ±5% tolerance)
        length_difference = abs(translated_length - ref_protein_length)
        length_tolerance = max(1, int(ref_protein_length * 0.05))

        if length_difference > length_tolerance:
            frameshift_mutations = True  # Large length mismatch
        else:
            frameshift_mutations = False  # Small variations are fine

    # **Fix: Only flag internal stop codons, not the last one**
    if '*' in protein_str[:-1]:  # Ignore the last stop codon
        frameshift_mutations = True

    return protein_str, frameshift_mutations


In [19]:
def align_and_handle_deletions(translated_seq, ref_seq):
    """Handle deletions and ensure proper alignment with placeholders for gaps."""
    aligned_seq = []
    ref_index = 0
    trans_index = 0

    while ref_index < len(ref_seq) and trans_index < len(translated_seq):
        if translated_seq[trans_index] == ref_seq[ref_index]:
            # If they match, add the amino acid from the translated sequence
            aligned_seq.append(translated_seq[trans_index])
            trans_index += 1
        else:
            if translated_seq[trans_index] == '-':
                # If there is a gap in the translated sequence, mark it as a gap
                aligned_seq.append('-')
                trans_index += 1
            elif ref_seq[ref_index] == '-':
                # If the reference has a gap, skip the reference gap
                ref_index += 1
            else:
                # If it's a substitution (mismatch), append the amino acid from the translated sequence
                aligned_seq.append(translated_seq[trans_index])
                trans_index += 1

        # Move the reference index forward regardless
        ref_index += 1

    # **Fix: Trim excess gaps at the end**
    # aligned_seq_str = ''.join(aligned_seq).rstrip('-')
    aligned_seq_str = ''.join(aligned_seq)

    return aligned_seq_str


In [20]:
all_translations = []
all_labels = []
problematic_indices = set()
frameshift_mutations_list = []

for i in range(len(gene_sequence_data)):
    problematic = False
    h37rv_aa_str = Seq(h37rv_sequence_str).translate(to_stop=True)
    reference_length = len(h37rv_aa_str)
    # Correct for CDS
    translation, frameshift = translate_sequence_with_gaps(gene_sequence_data["Sequence"][i], to_stop=False, ref_protein_length=reference_length)
    # dna fasta files already took care of orientation so we skip that step here.
    aligned_translation = align_and_handle_deletions(translation, h37rv_aa_str)


    if i == 1:
        print("translation before aligning", translation)
        print("translation after aligning", aligned_translation)
        print(f"Translated length: {len(translation)}, Reference length: {reference_length}")
        print(f"Final aligned sequence: {aligned_translation}")

    if frameshift:
        problematic = True
        problematic_indices.add(i)
        frameshift_mutations_list.append(1)
        aligned_translation = '0' * reference_length
    else:
        frameshift_mutations_list.append(0)

    # print(f"Row {row_index}: frameshift={frameshift}, aligned_translation={aligned_translation[:50]}...")
    all_translations.append(aligned_translation)
    all_labels.append(gene_sequence_data["Phenotype"][i])

# **Add new columns to the DataFrame**
gene_sequence_data["Protein_Sequence"] = all_translations
gene_sequence_data["Frameshift_Mutation"] = frameshift_mutations_list

translation before aligning VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLIWEVVDNAVDEAMAGYATTVNVVLLEDGGVEVADDGRGIPVATHASGIPTVDVVMTQLHAGGKFDSDAYAISGGLHGVGVSVVNALSTRLEVEIKRDGYEWSQVYEKSEPLGLKQGAPTKKTGSTVRFWADPAVFETTEYDFETVARRLQEMAFLNKGLTINLTDERVTQDEVVDEVVSDVAEAPKSASERAAESTAPHKVKSRTFHYPGGLVDFVKHINRTKNAIHSSIVDFSGKGTGHEVEIAMQWNAGYSESVHTFANTINTHEGGTHEEGFRSALTSVVNKYAKDRKLLKDKDPNLTGDDIREGLAAVISVKVSEPQFEGQTKTKLGNTEVKSFVQKVCNEQLTHWFEANPTDAKVVVNKAVSSAQARIAARKARELVRRKSATDIGGLPGKLADCRSTDPRKSELYVVEGDSAGGSAKSGRDSMFQAILPLRGKIINVEKARIDRVLKNTEVQAIITALGTGIHDEFDIGKLRYHKIVLMADADVDGQHISTLLLTLLFRFMRPLIENGHVFLAQPPLYKLKWQRSDPEFAYSDRERDGLLEAGLKAGKKINKEDGIQRYKGLGEMDAKELWETTMDPSVRVLRQVTLDDAAAADELFSILMGEDVDARRSFITRNAKDVRFLDV
translation after aligning VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLIWEVVDNAVDEAMAGYATTVNVVLLEDGGVEVADDGRGIPVATHASGIPTVDVVMTQLHAGGKFDSDAYAISGGLHGVGVSVVNALSTRLEVEIKRDGYEWSQVYEKSEPLGLKQGAPTKKTGSTVRFWADPAVFETTEYDFETVARRLQEMAFLNKGLTINLTDERVTQDEVVDEVVSDVAEAPKSASERAAESTAPHKVKSRTFHYPGGLVDFVKHINRTKN

### add protein sequence to the gene csv file and save

In [21]:
output_file_name=gene_name+"_"+ drug_name+ "_combined_sequence_data.csv"
gene_sequence_data.to_csv("../data/sequence_data_csv/"+output_file_name, index=False)

In [22]:
# Find longest matching subsequence (not just substring)
seq1 = gene_sequence_data['Protein_Sequence'][0] 
seq2 = ref_protein


matcher = SequenceMatcher(None, seq2, seq1)
match = matcher.find_longest_match(0, len(seq2), 0, len(seq1))

print(f"Longest match at ref_protein[{match.a}:{match.a + match.size}]")
print(f"Longest Match length: {match.size} / {len(seq1)} ({match.size / len(seq1):.2%})")

# Show matching region
print("Matched ref segment:", ref_protein[match.a:match.a + match.size])
print("Matched seq1 segment:", seq1[match.b:match.b + match.size])


print(f"Match ratio: {matcher.ratio() * 100:.2f}%")

Longest match at ref_protein[0:675]
Longest Match length: 675 / 675 (100.00%)
Matched ref segment: VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLIWEVVDNAVDEAMAGYATTVNVVLLEDGGVEVADDGRGIPVATHASGIPTVDVVMTQLHAGGKFDSDAYAISGGLHGVGVSVVNALSTRLEVEIKRDGYEWSQVYEKSEPLGLKQGAPTKKTGSTVRFWADPAVFETTEYDFETVARRLQEMAFLNKGLTINLTDERVTQDEVVDEVVSDVAEAPKSASERAAESTAPHKVKSRTFHYPGGLVDFVKHINRTKNAIHSSIVDFSGKGTGHEVEIAMQWNAGYSESVHTFANTINTHEGGTHEEGFRSALTSVVNKYAKDRKLLKDKDPNLTGDDIREGLAAVISVKVSEPQFEGQTKTKLGNTEVKSFVQKVCNEQLTHWFEANPTDAKVVVNKAVSSAQARIAARKARELVRRKSATDIGGLPGKLADCRSTDPRKSELYVVEGDSAGGSAKSGRDSMFQAILPLRGKIINVEKARIDRVLKNTEVQAIITALGTGIHDEFDIGKLRYHKIVLMADADVDGQHISTLLLTLLFRFMRPLIENGHVFLAQPPLYKLKWQRSDPEFAYSDRERDGLLEAGLKAGKKINKEDGIQRYKGLGEMDAKELWETTMDPSVRVLRQVTLDDAAAADELFSILMGEDVDARRSFITRNAKDVRFLDV
Matched seq1 segment: VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLIWEVVDNAVDEAMAGYATTVNVVLLEDGGVEVADDGRGIPVATHASGIPTVDVVMTQLHAGGKFDSDAYAISGGLHGVGVSVVNALSTRLEVEIKRDGYEWSQVYEKSEPLGLKQGAPTKKTGSTVRFWADPAVFETTEYDFETVARRLQEMAFLNKG

In [23]:
from Bio import pairwise2

alignments = pairwise2.align.globalxx(ref_protein, seq1)
top = alignments[0]
print(pairwise2.format_alignment(*top))


VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLIWEVVDNAVDEAMAGYATTVNVVLLEDGGVEVADDGRGIPVATHASGIPTVDVVMTQLHAGGKFDSDAYAISGGLHGVGVSVVNALSTRLEVEIKRDGYEWSQVYEKSEPLGLKQGAPTKKTGSTVRFWADPAVFETTEYDFETVARRLQEMAFLNKGLTINLTDERVTQDEVVDEVVSDVAEAPKSASERAAESTAPHKVKSRTFHYPGGLVDFVKHINRTKNAIHSSIVDFSGKGTGHEVEIAMQWNAGYSESVHTFANTINTHEGGTHEEGFRSALTSVVNKYAKDRKLLKDKDPNLTGDDIREGLAAVISVKVSEPQFEGQTKTKLGNTEVKSFVQKVCNEQLTHWFEANPTDAKVVVNKAVSSAQARIAARKARELVRRKSATDIGGLPGKLADCRSTDPRKSELYVVEGDSAGGSAKSGRDSMFQAILPLRGKIINVEKARIDRVLKNTEVQAIITALGTGIHDEFDIGKLRYHKIVLMADADVDGQHISTLLLTLLFRFMRPLIENGHVFLAQPPLYKLKWQRSDPEFAYSDRERDGLLEAGLKAGKKINKEDGIQRYKGLGEMDAKELWETTMDPSVRVLRQVTLDDAAAADELFSILMGEDVDARRSFITRNAKDVRFLDV
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

/work/pi_annagreen_umass_edu/mahbuba/protein/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [24]:
## explore and confirm translated outputs for feature and matrix.
unique_elements, counts = np.unique(frameshift_mutations_list, return_counts=True)
value_counts_dict = dict(zip(unique_elements, counts))
print(value_counts_dict)

problem_percentage = (len(problematic_indices) / len(gene_sequence_data)) * 100
print(f"Problematic percentage: {problem_percentage}%")

print("Keeping all sequences.")
valid_indices = range(len(gene_sequence_data))

valid_translations = [(filenames[i], all_translations[i][:reference_length]) for i in valid_indices]
valid_labels = [all_labels[i] for i in valid_indices]

valid_lengths = [len(seq[1]) for seq in valid_translations]
length_mismatches = [i for i, length in enumerate(valid_lengths) if length != reference_length]
total_sequences = len(valid_translations)
num_mismatches = len(length_mismatches)
print(f"Reference length expected: {reference_length}")
print(f"Sample of sequence lengths after truncation: {[len(seq[1]) for seq in valid_translations[:10]]}")

if num_mismatches == 0:
    print("All translations have the correct length.")
else:
    mismatch_percentage = (num_mismatches / total_sequences) * 100
    print(f"{num_mismatches} sequences have varying lengths out of {total_sequences} sequences.")
    print(f"Percentage of sequences with varying lengths: {mismatch_percentage:.2f}%")

print(f"Total sequences after filtering: {len(valid_translations)}")
print(f"Sample of sequence lengths after filtering: {[len(seq[1]) for seq in valid_translations[:10]]}")

{0: 71}
Problematic percentage: 0.0%
Keeping all sequences.
Reference length expected: 675
Sample of sequence lengths after truncation: [675, 675, 675, 675, 675, 675, 675, 675, 675, 675]
All translations have the correct length.
Total sequences after filtering: 71
Sample of sequence lengths after filtering: [675, 675, 675, 675, 675, 675, 675, 675, 675, 675]


In [25]:
#check with ref protein again
start_index = ref_protein.find(gene_sequence_data['Protein_Sequence'][0])
start_index

0

In [37]:
gene_sequence_data

,Filename,Sequence,Phenotype,seq_len,hamming_dist,Protein_Sequence,Frameshift_Mutation
0,SAMN08892550,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,R,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
1,SAMN08892544,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,R,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
2,SAMN08892578,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,S,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
3,SAMN08629278,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,S,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
4,SAMN08892419,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,R,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
...,...,...,...,...,...,...,...
66,SAMN08892417,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,R,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
67,SAMN08892574,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,S,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
68,SAMN08892576,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,R,2028,1,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0
69,SAMN08892562,GTGGCTGCCCAGAAAAAGAAGGCCCAAGACGAATACGGCGCTGCGT...,S,2028,0,VAAQKKKAQDEYGAASITILEGLEAVRKRPGMYIGSTGERGLHHLI...,0


# Step 6: write aa fasta file with filename, aa seq, phenotype and frameshift flag

In [26]:

def write_fasta_with_metadata_from_df(df, output_file, reference_length):
    """
    Writes translated protein sequences to a FASTA file with metadata in the header.

    Args:
        df (pd.DataFrame): DataFrame containing Filename, Protein_Sequence, Phenotype, and Frameshift_Mutation.
        output_file (str): Path to save the FASTA file.
        reference_length (int): Length of the reference protein sequence.
    """
    with open(output_file, "w") as fasta_file:
        for _, row in df.iterrows():
            filename = row["Filename"]
            sequence = row["Protein_Sequence"] 
            phenotype = row["Phenotype"]
            frameshift_flag = row["Frameshift_Mutation"]
            seq_len = row["seq_len"]
            

            # Construct FASTA header
            header = f">{filename} | {phenotype} | {seq_len} | Frameshift: {frameshift_flag}"
            fasta_file.write(header + "\n")
            fasta_file.write(sequence + "\n")


In [27]:
# Define output file path
protein_name = f'../data/aa_fasta/all_sequences_{gene_name}_{drug_name}_aa.fasta'

# Ensure the DataFrame has the correct columns
if {"Filename", "Protein_Sequence", "Phenotype", "Frameshift_Mutation"}.issubset(gene_sequence_data.columns):
    write_fasta_with_metadata_from_df(gene_sequence_data, protein_name, reference_length)
    print(f"FASTA file saved at {protein_name}")
else:
    print("Error: The DataFrame is missing required columns.")


FASTA file saved at ../data/aa_fasta/all_sequences_gyrB_LEVOFLOXACIN_aa.fasta


In [28]:

filtered_translations = [(fn, seq) for (fn, seq) in valid_translations if len(seq) == reference_length]

filtered_labels = [valid_labels[i] for i, (_, seq) in enumerate(valid_translations) if len(seq) == reference_length]

filtered_flags = [frameshift_mutations_list[i] for i, (_, seq) in enumerate(valid_translations) if len(seq) == reference_length]


print(f"Reference length expected: {reference_length}")
print(f"Total sequences before filtering: {len(all_translations)}")
print(f"Total sequences after filtering: {len(filtered_translations)}")
print(f"Dropped {len(all_translations) - len(filtered_translations)} sequences.")



Reference length expected: 675
Total sequences before filtering: 71
Total sequences after filtering: 71
Dropped 0 sequences.


['R',
 'R',
 'S',
 'S',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'S',
 'S',
 'S',
 'S',
 'R',
 'S',
 'S',
 'S',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'S',
 'R',
 'S',
 'R',
 'R',
 'R',
 'R',
 'S',
 'R',
 'S',
 'R',
 'S',
 'R',
 'R',
 'R',
 'R',
 'R',
 'S',
 'S',
 'R',
 'R',
 'R',
 'S',
 'R',
 'R',
 'S',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'R',
 'S',
 'R',
 'S',
 'R',
 'R',
 'S',
 'R',
 'S',
 'S']

# Step 7: One-hot encode mutated positions relative to reference protein
## Mutation = 1, match = 0

In [29]:
def convert_to_onehot_with_reference(aa_seq, ref_aa):
    # return np.array([1 if aa == ref else 0 for aa, ref in zip(aa_seq, ref_aa)])
    return np.array([0 if aa == ref else 1 for aa, ref in zip(aa_seq, ref_aa)])

def encode_sequence(sequence, reference_length, h37rv_aa_str):
    encoded = convert_to_onehot_with_reference(str(sequence), str(h37rv_aa_str))
    return encoded

In [30]:

# --------- Step 2: Encode Sequences ---------

num_cores = joblib.cpu_count()

encoded_features = Parallel(n_jobs=num_cores)(
    delayed(encode_sequence)(seq, reference_length, h37rv_aa_str)
    for _, seq in filtered_translations
)

# Sanity check
lengths = [len(f) for f in encoded_features]
assert all(l == reference_length for l in lengths), "Inconsistent feature lengths after encoding!"
print(f"Encoding complete. Unique encoded lengths: {set(lengths)}")


Encoding complete. Unique encoded lengths: {675}


# Step 8: Save feature matrix for ML training
## Shape: (n_samples, 1 + L), where L = reference protein length
## First column = binary mutation flag (e.g. frame-shift)

In [31]:

# --------- Step 3: Assemble Feature Matrix ---------

mutation_flags = np.array(filtered_flags).reshape(-1, 1)
encoded_matrix = np.array(encoded_features)
feature_matrix = np.column_stack((mutation_flags, encoded_matrix))

print(f"Final feature matrix shape: {feature_matrix.shape}")

# --------- Step 4: Save ---------

np.save(f'../data/feature_matrix_labels/{gene_name}_{drug_name}_feature_matrix.npy', feature_matrix)
np.save(f'../data/feature_matrix_labels/{gene_name}_{drug_name}_labels.npy', filtered_labels)
print("Full preprocessing pipeline complete.")

Final feature matrix shape: (71, 676)
Full preprocessing pipeline complete.


# Step 9: Sample Data loading for Model Training

In [32]:
X, y = load_feature_matrix_and_labels(gene_name)

/work/pi_annagreen_umass_edu/mahbuba/Data-Curation-for-MTB/protein-tasks/data/feature_matrix_labels/gyrB_feature_matrix.npy
Loading feature matrix and labels for gyrB from disk.


In [33]:
# try with full sequence first
y_encoded = encode_labels(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
print(f"Train data shape before deduplication: {np.array(X_train).shape}")
full_train_shape = np.array(X_train).shape


lasso = LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100])
# lasso = LassoCV()
lasso.fit(X_train,y_train)

regular_lasso_R2 = lasso.score(X_test, y_test)
print("Lasso Score", regular_lasso_R2)
# logging.info(f"Lasso Score: {regular_lasso_R2}")
regular_lasso_auc =sklearn.metrics.roc_auc_score(y_test, lasso.predict(X_test))
print("Lasso AUC", regular_lasso_auc)
# logging.info(f"Lasso AUC: {regular_lasso_auc}")
regular_lasso_mse = mean_squared_error(y_test, lasso.predict(X_test))
print(f"regular lasso mse:{regular_lasso_mse}")


ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10]).fit(X_train, y_train)
full_ridge_score = ridge.score(X_test, y_test)
full_ridge_auc = sklearn.metrics.roc_auc_score(y_test, ridge.predict(X_test))
full_ridge_mse = mean_squared_error(y_test, ridge.predict(X_test))
print("ridge score", full_ridge_score)
print("ridge mse", full_ridge_mse)
print("ridge auc", full_ridge_auc)

logistic_model = LogisticRegressionCV(cv=3, scoring="roc_auc", max_iter=5000,Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100], class_weight="balanced").fit(X_train, y_train)
log_score = logistic_model.score(X_test, y_test)
log_auc = sklearn.metrics.roc_auc_score(y_test, logistic_model.predict(X_test))
log_mse = mean_squared_error(y_test, logistic_model.predict(X_test))

print("log score", log_score)
print("log mse", log_mse)
print("log auc", log_auc)


Train data shape before deduplication: (56, 676)
Lasso Score -0.0006377551020408934
Lasso AUC 0.5
regular lasso mse:0.22236394557823133
ridge score -0.011224314196856877
ridge mse 0.22471651426596823
ridge auc 0.45
log score 0.45
log mse 0.6666666666666666
log auc 0.5


## generate combined matrix

In [34]:
import numpy as np
import pandas as pd
import os
from Bio.Seq import Seq
from difflib import SequenceMatcher
from joblib import Parallel, delayed

# -----------------------------
# CONFIGURATION
# -----------------------------
DATA_DIR = "../data/sequence_data_csv"
OUT_DIR = "../data/feature_matrix_labels"
REFERENCE_SEQ_FILE = "../data/catalog/protein_sequences.csv"

# -----------------------------
# UTILITIES
# -----------------------------
def hamming_distance(seq1, seq2):
    return sum(c1 != c2 for c1, c2 in zip(seq1, seq2))

def convert_to_onehot_with_reference(aa_seq, ref_aa):
    return np.array([0 if aa == ref else 1 for aa, ref in zip(aa_seq, ref_aa)])

def encode_sequence(sequence, reference):
    return convert_to_onehot_with_reference(str(sequence), str(reference))

def get_common_identifiers(gene_dfs):
    shared = set(gene_dfs[0]['Filename'])
    for df in gene_dfs[1:]:
        shared &= set(df['Filename'])
    return list(shared)

# -----------------------------
# MAIN PIPELINE
# -----------------------------
def generate_combined_feature_matrix(gene_names, output_prefix):
    ref_seqs = pd.read_csv(REFERENCE_SEQ_FILE)
    gene_dfs = []
    ref_proteins = {}

    # Load data per gene
    for gene in gene_names:
        file_path = os.path.join(DATA_DIR, f"{gene}_combined_sequence_data.csv")
        df = pd.read_csv(file_path)
        df = df[df['Frameshift_Mutation'] == 0].copy()
        gene_dfs.append(df)

        ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene, 'protein_sequence'].values[0]
        ref_proteins[gene] = ref_protein

    # Identify shared isolates
    common_ids = get_common_identifiers(gene_dfs)
    print(f"Number of shared isolates: {len(common_ids)}")

    all_vectors = []
    all_labels = []

    for idx in common_ids:
        row_vectors = []
        skip = False
        for i, gene in enumerate(gene_names):
            df = gene_dfs[i]
            row = df[df['Filename'] == idx].iloc[0]
            protein_seq = row['Protein_Sequence']
            ref_protein = ref_proteins[gene]
            try:
                encoded = np.asarray(encode_sequence(protein_seq, ref_protein)).flatten()
                if encoded.ndim != 1:
                    raise ValueError("Encoded vector is not 1D.")
                row_vectors.append(encoded)
            except Exception as e:
                print(f"Skipping {idx} due to vector problem in gene {gene}: {e}")
                skip = True
                break
        if skip:
            continue
        try:
            full_vector = np.concatenate(row_vectors)
            all_vectors.append(full_vector)
            all_labels.append(row['Phenotype'])
        except Exception as e:
            print(f"Skipping {idx} due to concatenation failure: {e}")

    expected_len = len(all_vectors[0])
    filtered = [(v, l) for v, l in zip(all_vectors, all_labels) if len(v) == expected_len]
    all_vectors, all_labels = zip(*filtered)
    X = np.stack(all_vectors)
    y = pd.Series(all_labels).astype("category").cat.codes.values


    # Save to disk
    os.makedirs(OUT_DIR, exist_ok=True)
    np.save(os.path.join(OUT_DIR, f"{output_prefix}_feature_matrix.npy"), X)
    np.save(os.path.join(OUT_DIR, f"{output_prefix}_labels.npy"), y)

    print(f"Saved combined matrix: {X.shape}, labels: {y.shape}")
    return X, y


In [7]:

genes = ["embA","embB", "embC"]  # or more
generate_combined_feature_matrix(genes, output_prefix="embA_embB_embC")

Number of shared isolates: 11954
Saved combined matrix: (11954, 3286), labels: (11954,)


(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 array([1, 1, 1, ..., 1, 1, 1], dtype=int8))